# PCA and Manifold Learning Visual Lab

Compare linear and nonlinear low-dimensional views of high-dimensional observations.

**Portfolio category:** Dimensionality reduction

**Data mode:** Built-in dataset

This notebook keeps labels out of fitting wherever labels exist, uses deterministic seeds,
reports unsupervised-specific diagnostics, and avoids hard-coded results.

## 1. Project setup

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)

## 2. Load high-dimensional digit data

In [ ]:
digits = load_digits()
sample_index = rng.choice(len(digits.data), size=1100, replace=False)
X_raw = digits.data[sample_index]
hidden_labels = digits.target[sample_index]
X = StandardScaler().fit_transform(X_raw)
print("Working shape:", X.shape)

## 3. Data quality and pixel variance

In [ ]:
pixel_variance = X_raw.var(axis=0)
display(pd.Series(pixel_variance).describe().to_frame("pixel_variance"))
print("Zero-variance pixels:", int((pixel_variance == 0).sum()))

## 4. PCA variance trade-off

In [ ]:
full_pca = PCA(random_state=RANDOM_STATE).fit(X)
cumulative = np.cumsum(full_pca.explained_variance_ratio_)
components_90 = int(np.searchsorted(cumulative, 0.90) + 1)
plt.plot(np.arange(1, len(cumulative) + 1), cumulative)
plt.axhline(0.90, color="red", linestyle="--")
plt.axvline(components_90, color="red", linestyle="--")
plt.title("Cumulative explained variance")
plt.xlabel("Components")
plt.ylabel("Explained variance")
plt.tight_layout()

## 5. Build PCA and t-SNE projections

In [ ]:
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X)
tsne_2d = TSNE(
    n_components=2,
    perplexity=30,
    init="pca",
    learning_rate="auto",
    random_state=RANDOM_STATE,
).fit_transform(X)

## 6. Neighbourhood preservation

In [ ]:
display(pd.Series({
    "pca_trustworthiness": trustworthiness(X, pca_2d, n_neighbors=10),
    "tsne_trustworthiness": trustworthiness(X, tsne_2d, n_neighbors=10),
    "pca_2d_explained_variance": PCA(n_components=2).fit(X).explained_variance_ratio_.sum(),
}).to_frame("value"))

## 7. Visual comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(pca_2d[:, 0], pca_2d[:, 1], c=hidden_labels, cmap="tab10", s=12)
axes[0].set_title("PCA")
axes[1].scatter(tsne_2d[:, 0], tsne_2d[:, 1], c=hidden_labels, cmap="tab10", s=12)
axes[1].set_title("t-SNE")
plt.tight_layout()

## 8. Downstream clustering comparison

In [ ]:
X_pca_90 = PCA(n_components=components_90, random_state=RANDOM_STATE).fit_transform(X)
raw_clusters = KMeans(n_clusters=10, n_init=30, random_state=RANDOM_STATE).fit_predict(X)
pca_clusters = KMeans(n_clusters=10, n_init=30, random_state=RANDOM_STATE).fit_predict(X_pca_90)
display(pd.Series({
    "raw_space_silhouette": silhouette_score(X, raw_clusters),
    "pca_90_space_silhouette": silhouette_score(X_pca_90, pca_clusters),
}).to_frame("value"))

## 9. Key findings

PCA supports reproducible global structure; t-SNE is better for local visual exploration and should not be interpreted as cluster proof.

## 10. Interpretation and responsible use

Treat the output as exploratory evidence, not ground truth. For pca and manifold learning visual lab,
validate stability on newer data, inspect edge cases, and review domain risks before
turning clusters, rankings or anomaly scores into decisions.

## 11. Next steps

- Replace demonstration data with a versioned, licensed dataset.
- Track data quality, drift and stability across repeated runs.
- Add domain-specific review before deployment.
- Package inference only after reproducibility and privacy checks pass.

All numeric results are generated at execution time; none are hard-coded.